# 03 Data Cleaning Daily Household Transactions

Notebook ini fokus hanya pada dataset `Daily Household Transactions.csv`.

Tujuan notebook:
- Membaca dan membedah struktur data daily household transaction.
- Membersihkan tanggal, teks, kategori, amount, duplikasi, dan missing value.
- Menandai outlier amount dengan metode IQR tanpa menghilangkan informasi original.
- Membuat data clean yang rapi untuk analisis household finance.
- Menyimpan hasil akhir ke folder `data/clean`.

Catatan cleaning utama:
- Semua transaksi tetap disimpan: `Expense`, `Income`, dan `Transfer-Out`.
- Amount original dalam INR tetap disimpan.
- Ditambahkan amount dalam IDR memakai asumsi `1 INR = 183 IDR` agar konsisten dengan notebook cleaning gabungan.
- Untuk analisis arus kas, dibuat `amount_signed_inr` dan `amount_signed_idr`: income bernilai positif, expense/transfer-out bernilai negatif.


## 1. Import library

Library dibuat sederhana: `pandas`, `numpy`, dan `matplotlib`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')


## 2. Konfigurasi path

Data raw dibaca dari `data/raw/daily_household_transaction`. Hasil clean disimpan ke `data/clean`.


In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

raw_path = project_root / 'data' / 'raw' / 'daily_household_transaction' / 'Daily Household Transactions.csv'
clean_path = project_root / 'data' / 'clean'
clean_path.mkdir(parents=True, exist_ok=True)

print('Project root :', project_root)
print('Raw file     :', raw_path)
print('Clean folder :', clean_path)

if not raw_path.exists():
    raise FileNotFoundError(f'File tidak ditemukan: {raw_path}')


## 3. Load data raw


❌❌DISINI UBAH INR KE IDR DAN JUGA UBAH TIPE DATANYA

In [ ]:
df_raw = pd.read_csv(raw_path)

print('Shape raw:', df_raw.shape)
display(df_raw.head(30))
display(pd.DataFrame({'column': df_raw.columns, 'dtype': df_raw.dtypes.astype(str).values}))


## 4. Bedah kualitas data awal

Cell ini mengecek missing value, duplikasi, tipe transaksi, currency, dan ringkasan amount.

❌HANDLE DATA YANG HILANG

In [ ]:
missing_df = (
    df_raw.isna().sum()
    .reset_index(name='missing_count')
    .rename(columns={'index': 'column'})
)
missing_df['missing_pct'] = (missing_df['missing_count'] / len(df_raw) * 100).round(2)

duplicate_exact = int(df_raw.duplicated().sum())
amount_desc = df_raw['Amount'].describe().reset_index()
amount_desc.columns = ['metric', 'value']

print('Exact duplicate rows:', duplicate_exact)
print('Jumlah amount <= 0:', int((df_raw['Amount'] <= 0).sum()))

display(missing_df)
display(df_raw['Income/Expense'].value_counts(dropna=False).reset_index(name='rows'))
display(df_raw['Currency'].value_counts(dropna=False).reset_index(name='rows'))
display(amount_desc)


In [ ]:
missing_df = (
    df_raw.isna().sum()
    .reset_index(name='missing_count')
    .rename(columns={'index': 'column'})
)
missing_df['missing_pct'] = (
    missing_df['missing_count'] / len(df_raw) * 100
).round(2)

duplicate_exact = int(df_raw.duplicated().sum())
amount_desc = df_raw['Amount'].describe().reset_index()
amount_desc.columns = ['metric', 'value']

# Ambil data yang Subcategory atau Note kosong
missing_subcat_note = df_raw[
    df_raw['Subcategory'].isna() | df_raw['Note'].isna()
][['Date', 'Category', 'Subcategory', 'Note', 'Amount']].head(10)

print('Exact duplicate rows:', duplicate_exact)
print('Jumlah amount <= 0:', int((df_raw['Amount'] <= 0).sum()))

display(missing_df)
display(df_raw['Income/Expense'].value_counts(dropna=False).reset_index(name='rows'))
display(df_raw['Currency'].value_counts(dropna=False).reset_index(name='rows'))
display(amount_desc)

print('\n10 baris data dengan Subcategory atau Note yang hilang:')
display(missing_subcat_note)

## 6. Helper cleaning

Helper dibuat pendek supaya logic cleaning mudah dibaca.


In [ ]:
INR_TO_IDR = 183


def clean_text(value, default='unknown'):
    if pd.isna(value):
        return default
    value = str(value).strip()
    return value if value else default


def parse_date(series):
    return pd.to_datetime(series, errors='coerce', dayfirst=True, format='mixed')


def standard_category(category):
    text = str(category).lower().strip()

    if text in ['food']:
        return 'Makanan'
    if text in ['transportation', 'tourism']:
        return 'Transportasi'
    if text in ['household', 'maid', 'garbage disposal', 'water (jar /tanker)', 'cook']:
        return 'Rumah Tangga'
    if text in ['subscription', 'rent', 'life insurance']:
        return 'Tagihan & Langganan'
    if text in ['health', 'beauty', 'grooming']:
        return 'Kesehatan'
    if text in ['education', 'self-development', 'culture', 'documents']:
        return 'Pendidikan & Dokumen'
    if any(word in text for word in ['mutual fund', 'investment', 'deposit', 'provident', 'share market', 'small cap', 'fixed deposit']):
        return 'Investasi & Tabungan'
    if text in ['salary', 'bonus', 'interest', 'dividend earned on shares', 'gpay reward', 'tax refund', 'amazon pay cashback', 'maturity amount', 'scrap']:
        return 'Pendapatan'
    if text in ['money transfer', 'saving bank account 1', 'saving bank account 2']:
        return 'Transfer'
    if text in ['apparel', 'gift', 'festivals']:
        return 'Belanja & Hadiah'
    if text in ['family']:
        return 'Keluarga'
    if text in ['social life']:
        return 'Sosial'
    return 'Lainnya'


def direction(transaction_type):
    if transaction_type == 'Income':
        return 1
    return -1


def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return q1, q3, iqr, lower, upper


## 7. Cleaning data

Langkah cleaning:
- Rename kolom ke format snake_case.
- Parse tanggal ke `datetime`.
- Bersihkan teks kosong pada `subcategory` dan `note`.
- Hapus exact duplicate.
- Buat kategori standar.
- Buat amount INR, IDR, dan signed amount untuk cashflow.
- Tandai outlier amount berdasarkan IQR.


In [ ]:
df_clean = df_raw.copy()

df_clean = df_clean.rename(columns={
    'Date': 'transaction_date',
    'Mode': 'payment_mode',
    'Category': 'category_raw',
    'Subcategory': 'subcategory_raw',
    'Note': 'note',
    'Amount': 'amount_inr',
    'Income/Expense': 'transaction_type',
    'Currency': 'currency',
})

before_duplicate = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
duplicate_removed = before_duplicate - len(df_clean)

df_clean['transaction_date'] = parse_date(df_clean['transaction_date'])
df_clean['payment_mode'] = df_clean['payment_mode'].map(clean_text)
df_clean['category_raw'] = df_clean['category_raw'].map(clean_text)
df_clean['subcategory_raw'] = df_clean['subcategory_raw'].map(clean_text)
df_clean['note'] = df_clean['note'].map(lambda value: clean_text(value, default=''))
df_clean['transaction_type'] = df_clean['transaction_type'].map(clean_text)
df_clean['currency'] = df_clean['currency'].map(clean_text)

df_clean['amount_inr'] = pd.to_numeric(df_clean['amount_inr'], errors='coerce').astype(float)
df_clean['amount_idr'] = df_clean['amount_inr'] * INR_TO_IDR
df_clean['category_clean'] = df_clean['category_raw'].map(standard_category)
df_clean['direction'] = df_clean['transaction_type'].map(direction)
df_clean['amount_signed_inr'] = df_clean['amount_inr'] * df_clean['direction']
df_clean['amount_signed_idr'] = df_clean['amount_idr'] * df_clean['direction']

df_clean['year'] = df_clean['transaction_date'].dt.year
df_clean['month'] = df_clean['transaction_date'].dt.month
df_clean['year_month'] = df_clean['transaction_date'].dt.to_period('M').astype(str)

q1, q3, iqr, lower_bound, upper_bound = iqr_bounds(df_clean['amount_inr'])
df_clean['is_outlier_amount'] = (df_clean['amount_inr'] < lower_bound) | (df_clean['amount_inr'] > upper_bound)
df_clean['amount_inr_capped'] = df_clean['amount_inr'].clip(lower=max(lower_bound, 0), upper=upper_bound)
df_clean['amount_idr_capped'] = df_clean['amount_inr_capped'] * INR_TO_IDR

df_clean = df_clean.sort_values('transaction_date').reset_index(drop=True)

display(df_clean.head())
print('Duplicate removed:', duplicate_removed)
print('IQR lower bound:', round(lower_bound, 2))
print('IQR upper bound:', round(upper_bound, 2))
print('Outlier amount:', int(df_clean['is_outlier_amount'].sum()))


## 8. Validasi hasil cleaning

Data dianggap rapi jika kolom penting tidak kosong: tanggal, amount, kategori, tipe transaksi, payment mode, dan currency.


In [ ]:
important_columns = ['transaction_date', 'amount_inr', 'category_raw', 'category_clean', 'transaction_type', 'payment_mode', 'currency']

validation_df = pd.DataFrame({
    'column': important_columns,
    'missing_count': [int(df_clean[column].isna().sum()) for column in important_columns],
})
validation_df['missing_pct'] = (validation_df['missing_count'] / len(df_clean) * 100).round(2)

summary_cleaning = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(df_raw)},
    {'metric': 'clean_rows', 'value': len(df_clean)},
    {'metric': 'duplicate_removed', 'value': duplicate_removed},
    {'metric': 'missing_subcategory_filled', 'value': int(df_raw['Subcategory'].isna().sum())},
    {'metric': 'missing_note_filled', 'value': int(df_raw['Note'].isna().sum())},
    {'metric': 'outlier_amount_iqr', 'value': int(df_clean['is_outlier_amount'].sum())},
    {'metric': 'iqr_lower_bound_inr', 'value': round(lower_bound, 2)},
    {'metric': 'iqr_upper_bound_inr', 'value': round(upper_bound, 2)},
])

display(validation_df)
display(summary_cleaning)

if validation_df['missing_count'].sum() > 0:
    raise ValueError('Masih ada missing value pada kolom penting.')

print('Validasi cleaning lolos.')


## 9. Analisis singkat setelah cleaning

Bagian ini membantu melihat data yang sudah rapi: tipe transaksi, kategori standar, dan outlier terbesar.


In [ ]:
display(df_clean['transaction_type'].value_counts().reset_index(name='rows'))
display(df_clean['category_clean'].value_counts().reset_index(name='rows'))

top_outlier_df = (
    df_clean[df_clean['is_outlier_amount']]
    .sort_values('amount_inr', ascending=False)
    .head(10)
    [['transaction_date', 'transaction_type', 'category_raw', 'category_clean', 'subcategory_raw', 'amount_inr', 'amount_idr', 'note']]
)
display(top_outlier_df)


## 11. Dataset final

`df_clean` adalah satu dataset utama yang menyimpan semua transaksi household yang sudah rapi. Tipe transaksi tetap dibedakan lewat kolom `transaction_type`, jadi analisis expense, income, atau transfer bisa difilter dari satu file yang sama tanpa perlu menyimpan banyak dataset.


In [ ]:
final_columns = [
    'transaction_date', 'year', 'month', 'year_month',
    'transaction_type', 'direction', 'payment_mode', 'currency',
    'category_raw', 'category_clean', 'subcategory_raw', 'note',
    'amount_inr', 'amount_idr', 'amount_signed_inr', 'amount_signed_idr',
    'is_outlier_amount', 'amount_inr_capped', 'amount_idr_capped',
]

df_clean = df_clean[final_columns].copy()

display(df_clean.head())
print('Total clean rows:', len(df_clean))
print('Expense rows:', int((df_clean['transaction_type'] == 'Expense').sum()))
print('Income rows:', int((df_clean['transaction_type'] == 'Income').sum()))
print('Transfer-Out rows:', int((df_clean['transaction_type'] == 'Transfer-Out').sum()))


## 12. Simpan satu dataset utama ke data/clean

Hanya satu file utama yang disimpan: `daily_household_transactions_clean.csv`.

Summary dan validation tetap dibuat di notebook untuk pengecekan, tetapi tidak disimpan sebagai file terpisah agar folder `data/clean` tetap sederhana.


In [ ]:
clean_file = clean_path / 'daily_household_transactions_clean.csv'

df_clean.to_csv(clean_file, index=False)

print('Saved clean data:', clean_file)


## 13. Variabel penting

- `df_raw`: data asli.
- `df_clean`: satu dataset utama yang sudah rapi.
- `summary_cleaning`: ringkasan cleaning di memory notebook.
- `validation_df`: validasi kolom penting di memory notebook.
